## 선택 · 심화 문제 1. Canonical manifest와 config hash

### 문제 배경

같은 설정인데 dictionary 입력 순서만 달라 hash가 바뀌면 실험 식별자로 쓸 수 없습니다. JSON을 canonical form으로 만들고 설정·환경·산출물을 함께 기록합니다.

### 시작 코드

```python
config = {"seed": 42, "model_id": "mini-transformer", "dataset_version": "news-v1", "metric": "macro_f1"}

def build_manifest(config, environment):
    raise NotImplementedError
```

### 수행 요구사항

1. 네 필수 config key를 검사하세요.
2. `sort_keys=True`, UTF-8 JSON을 SHA-256으로 hash하고 앞 12자를 사용하세요.
3. 원본 dictionary를 수정하지 마세요.
4. Python/package version을 `environment`, checkpoint·prediction 경로를 `artifacts`에 넣으세요.

### 제출 결과

- manifest와 `config_hash`
- key 순서 변경 시 같은 hash, seed 변경 시 다른 hash 검증
- `심화 문제 1 자동 검증: PASS`

### 자동 검증

```python
env = {"python": "3.12", "torch": "2.x", "transformers": "5.14.1"}
m1 = build_manifest(config, env)
m2 = build_manifest(dict(reversed(list(config.items()))), env)
m3 = build_manifest({**config, "seed": 7}, env)
assert m1["config_hash"] == m2["config_hash"] != m3["config_hash"]
print("심화 문제 1 자동 검증: PASS")
```
    ```
    
  **상세 해설** · Hash는 동일 설정을 빠르게 비교하는 식별자일 뿐 동일 결과를 보장하지 않습니다. 데이터 내용, model revision, 비결정적 GPU 연산까지 통제하려면 별도 fingerprint와 환경 기록이 필요합니다.
    
  **자주 하는 실수**
    
    - `str(dict)`를 hash해 key 순서·표현에 의존합니다.
    - 원본 config에 hash를 삽입한 뒤 다시 hash해 값이 달라집니다.
    - Model ID만 기록하고 revision·tokenizer·dataset version을 빠뜨립니다.

---

## 최종 자동 검증 및 제출 체크

세 문제의 PASS를 확인하고, 실제로 실행한 설정과 생성된 경로만 아래 형식으로 정리하세요. 예시 값을 그대로 제출하지 않습니다.

```
1-2 | problem | dataset_version | split_seed | model/tokenizer | selection metric | test policy | config_hash | artifact path
```

---

In [2]:
import hashlib
import json

config = {"seed": 42, "model_id": "mini-transformer", "dataset_version": "news-v1", "metric": "macro_f1"}

def build_manifest(config, environment):
    # Hash 계산 전에 재현에 필요한 최소 key가 모두 있는지 검사합니다.
    required = {"seed", "model_id", "dataset_version", "metric"}
    missing = sorted(required - config.keys())
    if missing:
        raise ValueError(f"필수 설정 누락: {missing}")
    # Key 입력 순서와 공백 차이를 제거한 canonical JSON입니다.
    canonical = json.dumps(config, sort_keys=True, ensure_ascii=False, separators=(",", ":"))
    config_hash = hashlib.sha256(canonical.encode("utf-8")).hexdigest()[:12]
    # 원본 dictionary를 공유하지 않도록 config와 environment를 복사합니다.
    return {
        "config": dict(config),
        "environment": dict(environment),
        "artifacts": {"checkpoint": "artifacts/model", "predictions": "artifacts/test_predictions.jsonl"},
        "config_hash": config_hash,
    }

env = {"python": "3.12", "torch": "2.x", "transformers": "5.14.1"}
m1 = build_manifest(config, env)
m2 = build_manifest(dict(reversed(list(config.items()))), env)
m3 = build_manifest({**config, "seed": 7}, env)

print("hash:", m1["config_hash"])
print("same order-independent:", m1["config_hash"] == m2["config_hash"])
print("seed changes hash:", m1["config_hash"] != m3["config_hash"])
assert m1["config_hash"] == m2["config_hash"] != m3["config_hash"]
assert "config_hash" not in config
print("심화 문제 1 자동 검증: PASS")

hash: 97ce428b07a4
same order-independent: True
seed changes hash: True
심화 문제 1 자동 검증: PASS
